In [ ]:
import pandas as pd
basedir = r'E:\University of Michigan Dropbox\Jiahui Chen'
basedir = r'C:\Users\jiahuic\University of Michigan Dropbox\Jiahui Chen'
ctdf = pd.read_csv(basedir + r'\\PVEV\utility_matching\tempreg_withutility_usurdb_filled_sales_state.csv')
util_list = ctdf['matched_utility'].unique()


In [ ]:
import pandas as pd 
import numpy as np
import os
basedir = r'E:\University of Michigan Dropbox\Jiahui Chen'
# basedir = r'C:\Users\jiahuic\University of Michigan Dropbox\Jiahui Chen'
ctdf_lst = os.listdir(basedir + r'\\PVEV\\ctdfs')
combined_info = []
for file in ctdf_lst:
    if file.endswith('.csv'):
        temp_df = pd.read_csv(os.path.join(basedir + r'\\PVEV\ctdfs', file))
        temp_df['combined'] = temp_df['USPS'] + '_' + temp_df['matched_utility'].str.lower() + '_' + temp_df['name']+'__' + str(temp_df['urdb_label'].to_list()[0])
        combined_info.extend(temp_df['combined'].unique().tolist())
        if 'WI_wisconsin power & light co_Residential Time-of-Use (Three-Phase, Option D) Rg 2__664ba6b92b718f86d8004003' in temp_df['combined'].unique().tolist():
            print(file)
combined_info = list(set(combined_info))
combined_info = sorted(combined_info, key=lambda x: str(x).lower())
combined_info = [info for info in combined_info if isinstance(info, str) and info.strip()]
combined_info

In [ ]:
import os
import pandas as pd

# Define the directory containing the CSV files
ctdfs_dir = os.path.join(basedir, 'PVEV', 'ctdfs')

# List all CSV files in the directory
csv_files = [file for file in os.listdir(ctdfs_dir) if file.endswith('.csv')]

# Read and combine all CSV files into a single DataFrame
ctdfs = [pd.read_csv(os.path.join(ctdfs_dir, file)) for file in csv_files]
combined_ctdfs = pd.concat(ctdfs, ignore_index=True)
# Discard rows where 'Unnamed: 0.1' is missing
combined_ctdfs = combined_ctdfs[combined_ctdfs['Unnamed: 0.1'].notna()]
# Display the combined DataFrame
combined_ctdfs['combined'] = combined_ctdfs['USPS'] + '_' + combined_ctdfs['matched_utility'].str.lower() + '_' + combined_ctdfs['name']+'__' + combined_ctdfs['urdb_label']
combined_ctdfs.to_csv(basedir + r'\\PVEV\\utility_matching\Rates_dgrules.csv')

In [ ]:
# put in rule group to ctdf with util program names
ctdf = pd.read_csv(basedir + r'\\PVEV\\utility_matching\Rates_dgrules.csv')
for infoi in range(79):
    info = combined_info[infoi]
    if infoi in [0,1,2,3,4,5,8,11,12,15,20,23,24,29,30,31,32,33,37,38,39,40,41,44,45,51,53,60, 64, 65, 66, 67, 70,73,75,77]:
        dgrules = 'NEB'
    elif infoi in [6,7,9,10,13,14,16,17,18,21,28,34,46,47,48,49,54,55,56,58,59,62,63,69,71,78,55,72]:
        dgrules = 'NEM'
    elif infoi in [19,22,25,26,27,35,36,42,43,50,52,57,61,68,74,76]:
        dgrules = 'LMP'

    
    ctdf.loc[
        ctdf['combined'].str.lower() == info.lower(),
        'DG Rules'
    ] = dgrules
    # filtered_ctdf = ctdf[
    #     (ctdf['USPS'].str.lower() == info.split('_')[0].lower()) &
    #     (ctdf['matched_utility'].str.lower() == info.split('_')[1].lower()) &
    #     (ctdf['urdb_label'].str.lower() == info.split('__')[-1].lower())
    # ]
    # filtered_ctdf
ctdf_0 = pd.read_csv(basedir + r'\\PVEV\\utility_matching\tempreg_withutility_usurdb_filled_sales_state.csv')
ctdf = ctdf.set_index('fips').reindex(ctdf_0['fips']).reset_index()
ctdf.to_csv(basedir + r'\\PVEV\\utility_matching\Rates_dgrules.csv')

In [ ]:
ctdf.loc[ctdf['USPS'] == 'CA']

In [ ]:
basedir = r'C:\Users\jiahuic\University of Michigan Dropbox\Jiahui Chen'

import os 
import pandas as pd
from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)
# cost of electrification
import plotly.express as px
import plotly.io as pio
import plotly
# for iyear in ['24']:
ht = 350
wd = 700
ctdf = pd.read_csv(basedir + r'\\PVEV\\utility_matching\Rates_dgrules.csv')
uqlst = np.unique(ctdf['tempreg'])
ctdf['fips'] = ctdf.apply(lambda row: '0'+str(row['fips']) if len(str(row['fips']))==4 else row['fips'],axis=1)
uqlst = np.unique(ctdf['tempreg'])
ctdf = pd.concat([ctdf,ctdf.loc[ctdf[:].index[2330:2331]]])
ctdf['index'] = np.arange(ctdf.shape[0])
ctdf.set_index('index',inplace=True)
ctdf['fips'][ctdf.index[-1]]='46113'
fig = px.choropleth(ctdf, geojson=counties, locations='fips', color='DG Rules',width=700, height=320,projection='mercator'
                        )
fig.update_geos(fitbounds='locations',visible=False,lataxis_range=[26.84, 49.38],lonaxis_range=[-110,-65.95])
fig.update_layout(
    autosize=False,
    margin=dict(l=0,r=0,b=0,t=0,pad=0)
)# ,paper_bgcolor="LightSteelBlue"
fig.update_xaxes(automargin=True)
config = {
'toImageButtonOptions': {
    'format': 'png', # one of png, svg, jpeg, webp
    'height': ht,
    'width': wd,
    'scale':17 # Multiply title/legend/axis/canvas sizes by this factor
}
}
fig.update_traces(marker_line_width=0)
fig.show(config=config)
fig.write_image(basedir + r'\\PVEV\\Report\figs\DGRules.pdf',engine='kaleido')

In [ ]:
ctdf = pd.read_csv(basedir + r'\\PVEV\\utility_matching\Rates_dgrules_tou.csv')
for tempregi in range(432):
    urdb_label = ctdf.loc[ctdf['tempreg']==tempregi,'urdb_label'].to_list()[0]
    urdbdf = pd.read_csv(basedir + fr'\\PVEV\\julia\Utility_structure\{urdb_label}.csv')
    is_flat = urdbdf['energy_rates'].groupby(urdbdf.index // 24).apply(lambda x: x.nunique() == 1).all()
    # ctdf.loc[ctdf['tempreg'] == tempregi, 'flat_rate'] = is_flat
    ctdf.loc[ctdf['tempreg']==tempregi,'tou'] = ~is_flat
    urdbdf['tou'] = ~is_flat
    urdbdf.to_csv(basedir + fr'\\PVEV\\julia\Utility_structure\{urdb_label}.csv')
ctdf.to_csv(basedir + r'\\PVEV\\utility_matching\Rates_dgrules_tou.csv')

In [ ]:
import os 
from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)
# cost of electrification
import plotly.express as px
import plotly.io as pio
import plotly
# for iyear in ['24']:
ht = 350
wd = 700
ctdf = pd.read_csv(basedir + r'\\PVEV\\utility_matching\Rates_dgrules_tou.csv')
# ctdf['tou'] = ctdf['tou'].replace({1: 'TOU', 0: 'Flat'})
uqlst = np.unique(ctdf['tempreg'])
ctdf['fips'] = ctdf.apply(lambda row: '0'+str(row['fips']) if len(str(row['fips']))==4 else row['fips'],axis=1)
uqlst = np.unique(ctdf['tempreg'])
ctdf = pd.concat([ctdf,ctdf.loc[ctdf['Short name']=='SDJackson County']])
ctdf['index'] = np.arange(ctdf.shape[0])
ctdf.set_index('index',inplace=True)
ctdf['fips'][ctdf.index[-1]]='46113'
fig = px.choropleth(ctdf, geojson=counties, locations='fips', color='tou',width=700, height=320,projection='mercator'
                        )

fig.update_geos(fitbounds='locations',visible=False,lataxis_range=[26.84, 49.38],lonaxis_range=[-110,-65.95])
fig.update_layout(
    autosize=False,
    margin=dict(l=0,r=0,b=0,t=0,pad=0)
)# ,paper_bgcolor="LightSteelBlue"
fig.update_xaxes(automargin=True)
config = {
'toImageButtonOptions': {
    'format': 'png', # one of png, svg, jpeg, webp
    'height': ht,
    'width': wd,
    'scale':17 # Multiply title/legend/axis/canvas sizes by this factor
}
}
fig.update_traces(marker_line_width=0)
fig.show(config=config)
fig.write_image(basedir + r'\\PVEV\\Report\figs\TOU.png',format='png',engine='kaleido')

In [ ]:
ctdf.loc[ctdf['fips']==40083]

In [ ]:
ctdf.loc[ctdf['combined']=='NV_nevada power co_ORM-TOU Optional Multi-Family Residential Service Time of Use__66183766af1bea858b0b396c','tempreg'].unique()


In [ ]:
ctdf.loc[ctdf['tou']=='Flat','combined'].unique()

In [ ]:
index = combined_info.index('MS_entergy mississippi inc_Residential Service Three Phase (RS-38C)__668479c17ba7fc0e710f9f2d_missing.csv')
print(index)

In [ ]:
util_num = 78
urdb_labels = [info.split('__')[-1] for info in combined_info if isinstance(info, str)]
cleaned_filename = "".join(c for c in (combined_info[util_num] + '.csv') if c.isalnum() or c in (' ', '.', '_')).rstrip()
print(util_num, cleaned_filename)
if cleaned_filename in os.listdir(basedir + r'\\PVEV\PVcompensation\rates') or combined_info[util_num] + '.csv' in os.listdir(basedir + r'\\PVEV\PVcompensation\rates'):
    print('NEB')

net billing
0,1,2,3,4,5,8,11,12,15,20,23,24,29,30,31,32,33,37,38,39,40,41,44,45,51,53,60, 64, 65, 66, 67, 70,73,75，77


net metering: 
6 CO: https://www.xcelenergy.com/staticfiles/xe/Marketing/Files/CO-SR-Metering-and-Billing-FAQs.pdf
7 CT: https://www.eversource.com/content/docs/default-source/rates-tariffs/rider-n-class-1.pdf?sfvrsn=e31aa141_1
9 DE: https://www.delmarva.com/smart-energy/my-green-power-connection/developers-contractors/net-metering
10 FL: https://www.fpl.com/clean-energy/net-metering.html
13 IA: https://www.midamericanenergy.com/customer-interconnection
14 ID: https://www.myavista.com/energy-savings/green-options/go-solar/learn-about-net-metering
16 ID: https://www.rockymountainpower.net/content/dam/pcorp/documents/en/rockymountainpower/rates-regulation/idaho/rates/136_Net_Billing_Service.pdf
17 IL: https://www.ameren.com/illinois/residential/supply-choice/renewables/net-metering
18 IL: https://www.comed.com/smart-energy/my-green-power-connection/solar-energy/private-solar/solar-incentives-credits
21 KS: https://www.evergy.com/-/media/documents/billing/kansas-central/riders-and-surcharges/net-metering-rider.pdf
28 ME (NEM for res, NEB for com and ind): https://www.maine.gov/mpuc/regulated-utilities/electricity/neb
34 MT 64 SD northwestern energy: https://northwesternenergy.com/docs/default-source/default-document-library/billing-and-payment/rates-and-tariffs/montana/rules/electric/rule_16_et.pdf
46 NY https://lite.coned.com/_external/cerates/documents/elecPSC10/electric-tariff.pdf#page=283
47 NY https://www.psegliny.com/saveenergyandmoney/greenenergy/todnetmeter/netmetering
48 NY https://www.nationalgridus.com/solar-hub/Solar-for-Your-Home/Residential-Solar/Incentives-Programs#:~:text=Net%20Metering&text=If%20your%20solar%20panel%20system,on%20a%20future%20electric%20bill.
49 NY https://www.nationalgridus.com/solar-hub/Solar-for-Your-Home/Residential-Solar/Incentives-Programs#:~:text=Net%20Metering&text=If%20your%20solar%20panel%20system,on%20a%20future%20electric%20bill.
54 OR https://www.pacificpower.net/content/dam/pcorp/documents/en/pacificpower/rates-regulation/oregon/tariffs/rates/135_Net_Metering_Service_Optional_for_Qualifying_Customers.pdf
55 OR https://assets.ctfassets.net/416ywc1laqmd/6VdcgRq1GjVQIaOdRGuhm3/af780118ef83c16f4e3967c46fba1093/Sched_203.pdf
56 PA https://azure-na-assets.contentstack.com/v3/assets/blt1b5616c79bacadb4/bltc57c7aa2ea56c56c/65b42983835578000a5ef269/RS-2_Tariff_(1).pdf
58 PA_ppl electric utilities corp https://www.pplelectric.com/-/media/PPLElectric/At-Your-Service/Docs/Current-Electric-Tariff/net-metering-renewable.pdf
59 PA_ppl electric utilities corp https://www.pplelectric.com/-/media/PPLElectric/At-Your-Service/Docs/Current-Electric-Tariff/net-metering-renewable.pdf
62 SC DOMinion https://cdn-dominionenergy-prd-001.azureedge.net/-/media/pdfs/south-carolina/rates-and-tariffs/rider-to-retail-rates---third-net-energy-metering-for-renewable-energy-facilities.pdf?la=en&rev=2e9ee1a984884e35a1720571cf4d042c&hash=E159030D181ACAC07DE07377240948C8
63 VA DOMinion https://cdn-dominionenergy-prd-001.azureedge.net/-/media/pdfs/virginia/terms-and-conditions/vatc25ra.pdf?rev=225084daeb654d888e036bfff13ccfb4
69 VA DOMinion https://cdn-dominionenergy-prd-001.azureedge.net/-/media/pdfs/virginia/terms-and-conditions/vatc25ra.pdf?rev=225084daeb654d888e036bfff13ccfb4
71 WA puget sound energy inc file:///C:/Users/J/AppData/Local/Temp/MicrosoftEdgeDownloads/3bc58318-ebb6-44e7-bd99-a14dee232d56/elec_sch_150.pdf
78 WY https://www.rockymountainpower.net/content/dam/pcorp/documents/en/rockymountainpower/rates-regulation/wyoming/rates/135_Net_Metering_Service.pdf
Locational marginal
19 IN: https://www.duke-energy.com/-/media/pdfs/for-your-home/rates/electric-in/iurc-16/tariff-no-54-excess-distributed-generation.pdf?rev=96aa8227950147f3b95e1a5a8a2895aa
22 KY: https://psc.ky.gov/tariffs/Electric/Kentucky%20Power%20Company/Tariff.pdf
25 MA: https://www.eversource.com/content/docs/default-source/rates-tariffs/ma-electric/68-tariff-ma.pdf?sfvrsn=bfdde372_8
26 MD (also PJM capacity, neglected): https://azure-na-assets.contentstack.com/v3/assets/blt71bfe6e8a1c2d265/blt73f93e7df976b647/P3_SCH_X.pdf
27 MD (also PJM capacity, neglected): https://www.delmarva.com/cdn/assets/v3/assets/blt47b6e332b18fb457/bltefc1a0bec7ba2be7/67ed6ee63c65952497e53e9c/2025-03-05_MASTER_MD_DPL_Tariff_Book_1.pdf?branch=prod_alias
35 NC https://www.duke-energy.com/-/media/pdfs/for-your-home/rates/electric-in/iurc-16/tariff-no-54-excess-distributed-generation.pdf?rev=96aa8227950147f3b95e1a5a8a2895aa
36 NC, 62 SC https://www.duke-energy.com/-/media/pdfs/for-your-home/rates/electric-in/iurc-16/tariff-no-54-excess-distributed-generation.pdf?rev=96aa8227950147f3b95e1a5a8a2895aa
42 NJ (also PJM capacity, neglected):https://www.firstenergycorp.com/content/dam/customer/Customer%20Choice/Files/New%20Jersey/tariffs/BPU-14-Part-III-Eff-3-1-2025.pdf
43 NJ (also PJM capacity, neglected):https://www.firstenergycorp.com/content/dam/customer/Customer%20Choice/Files/New%20Jersey/tariffs/BPU-14-Part-III-Eff-3-1-2025.pdf
50 OH https://www.firstenergycorp.com/content/dam/customer/Customer%20Choice/Files/Ohio/tariffs/OE-2025-Electric-Service.pdf#page=165.99
52 OK https://www.oge.com/wps/wcm/connect/de61550b-57c0-478c-9681-b47bd00ef9da/70.10%20-%20NEBO%20-%20Stamped%20Approved.pdf?MOD=AJPERES&CACHEID=ROOTWORKSPACE-de61550b-57c0-478c-9681-b47bd00ef9da-oencMlS
57 PA https://view.officeapps.live.com/op/view.aspx?src=https%3A%2F%2Fwww.puc.pa.gov%2FPcDocs%2F568424.doc&wdOrigin=BROWSELINK
61  https://www.duke-energy.com/-/media/pdfs/for-your-home/rates/electric-in/iurc-16/tariff-no-54-excess-distributed-generation.pdf?rev=96aa8227950147f3b95e1a5a8a2895aa
68 VA LMP*1.11301 https://www.appalachianpower.com/lib/docs/ratesandtariffs/Virginia/APCoStandardTariff28-January-1-2025SUTUpdate.pdf
74 WI Time-of-day customers will have consumption
and production netted by pricing period under the rate schedule in which
the customer purchases energy.
76 WV https://www.appalachianpower.com/lib/docs/ratesandtariffs/WestVirginia/ENECTariffSheetsEff3-12-25.pdf

Tiered net metering
55 OR https://assets.ctfassets.net/416ywc1laqmd/6VdcgRq1GjVQIaOdRGuhm3/af780118ef83c16f4e3967c46fba1093/Sched_203.pdf
72 WI Peak offset peak, offpeak offset offpeak, carry over to next month, credit https://xcelnew.my.salesforce.com/sfc/p/#1U0000011ttV/a/8b000003BF6D/KWwCDj3uMqk2klcrd9kHb0WRgMm1s6O55xoIIp_FVPM

In [ ]:
len(combined_info)

NY phasing out NEM, substitute with VDER: https://www.nationalgridus.com/media/pdfs/billing-payments/electric-rates/upstate-ny/2025/stat_vder-cred_92_effective_april_30_2025.pdf


In [ ]:
util_num = 53

urdb_labels = [info.split('__')[-1] for info in combined_info if isinstance(info, str)]
print(combined_info[util_num])

In [ ]:
import os
import requests
from datetime import datetime
# Define the URL and the destination folder
url = r'https://www.psoklahoma.com/lib/docs/ratesandtariffs/Oklahoma/PSO_NEBO_Avoided_Energy_Prices_04-28-2025.pdf'
ratebooks_folder = basedir + r'\\PVEV\PVcompensation\ratebooks'
rate_folder = basedir + r'\\PVEV\PVcompensation\rates'

# Extract the filename from the URL
filename = combined_info[util_num] + '.pdf'  # Use the first item in combined_info as the filename

# Define the full path to save the PDF
file_path = os.path.join(ratebooks_folder, filename)

# Log the retrieval time
log_filename = combined_info[util_num]+".txt"
log_file_path = os.path.join(ratebooks_folder, log_filename)
# Remove illegal characters from the filename
filename = "".join(c for c in filename if c.isalnum() or c in (' ', '.', '_')).rstrip()
file_path = os.path.join(ratebooks_folder, filename)
log_filename = "".join(c for c in log_filename if c.isalnum() or c in (' ', '.', '_')).rstrip()
log_file_path = os.path.join(ratebooks_folder, log_filename)
with open(log_file_path, 'a') as log_file:
    log_file.write(f"{filename} retrieved at {datetime.now()}\nfrom {url}\n")
# Check if the URL points to a PDF or a webpage
if url.endswith('.pdf'):
    # Download the PDF
    response = requests.get(url)
    if response.status_code == 200:
        with open(file_path, 'wb') as f:
            f.write(response.content)
        print(f"PDF downloaded successfully to {file_path}")
    else:
        print(f"Failed to download PDF. Status code: {response.status_code}")
else:
    # Take a snapshot of the webpage as HTML
    response = requests.get(url)
    if response.status_code == 200:
        html_file_path = file_path.replace('.pdf', '.html')
        with open(html_file_path, 'w', encoding='utf-8') as f:
            f.write(response.text)
        print(f"Webpage snapshot saved successfully to {html_file_path}")
    else:
        print(f"Failed to retrieve webpage. Status code: {response.status_code}")


In [ ]:
# 53 OK 
# Define the time range for the year 2024
time_index = pd.date_range(start='2024-01-01 00:00:00', end='2024-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define the monthly avoided energy costs
monthly_rates = {
    1: 0.0601, 2: 0.0177, 3: 0.0205, 4: 0.0252, 5: 0.0272,
    6: {'on_peak': 0.0334, 'off_peak': 0.0289},
    7: {'on_peak': 0.0346, 'off_peak': 0.0292},
    8: {'on_peak': 0.0343, 'off_peak': 0.0304},
    9: {'on_peak': 0.0325, 'off_peak': 0.0268},
    10: {'on_peak': 0.0384, 'off_peak': 0.0346},
    11: 0.0287, 12: 0.0319
}

# Assign rates based on the defined periods
for month, rates in monthly_rates.items():
    if isinstance(rates, dict):  # On-peak and off-peak rates
        on_peak_hours = (
            (rate_series.index.month == month) &
            (rate_series.index.weekday < 5) &  # Monday to Friday
            (rate_series.index.hour >= 14) & (rate_series.index.hour < 19)  # 2 PM to 7 PM
        )
        rate_series[on_peak_hours] = rates['on_peak']*1.05*100
        rate_series[~on_peak_hours & (rate_series.index.month == month)] = rates['off_peak']*1.05*100
    else:  # Flat rate for the month
        rate_series[rate_series.index.month == month] = rates*1.05*100

# Convert the rate series to a DataFrame for better visualization
rate_df = rate_series.to_frame(name='Rate (cent/kWh)')
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 0  # Add a fixed rate column with default value 0

# Save the DataFrame to a CSV file
rate_df.to_csv(os.path.join(rate_folder, cleaned_filename), index=False)

In [ ]:
# 76 WV 
# Define the time range for the year
time_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='H')

# Create a constant rate series
rate_series = pd.Series(0.09343 * 100, index=time_index, name='Rate (cent/kWh)')  # Convert to cents

# Convert the series to a DataFrame for better visualization
rate_df = rate_series.to_frame()
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 0  # Add a fixed rate column with default value 0
rate_df.drop(columns=['Timestamp'], inplace=True)

# Save the DataFrame to a CSV file
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num] + '.csv'), index=False)

In [ ]:
# 75 WI_wisconsin public service corp_Residential Three-tier Time-of-Use Rg5-OTOU__6696dc3c28d79d9f7b06ad16
# Define the time range for the year
time_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='H')

# Create a constant rate series
rate_series = pd.Series(0.03141 * 100, index=time_index, name='Rate (cent/kWh)')  # Convert to cents

# Convert the series to a DataFrame for better visualization
rate_df = rate_series.to_frame()
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 0  # Add a fixed rate column with default value 0
rate_df.drop(columns=['Timestamp'], inplace=True)

# Save the DataFrame to a CSV file
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num] + '.csv'), index=False)

In [ ]:
# 73 WI wisconsin electric power co
# Define the time range for the year
util_rate = pd.read_csv(basedir + fr'\\PVEV\julia\Utility_structure\{urdb_labels[util_num]}.csv')
util_rate

time_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define the summer and non-summer months
summer_months = [6, 7, 8, 9]
non_summer_months = [1, 2, 3, 4, 5, 10, 11, 12]

# Assign rates for summer months
for month in summer_months:
    on_peak_hours = (rate_series.index.month == month) & (rate_series.index.hour >= 10) & (rate_series.index.hour < 22)
    rate_series[on_peak_hours] = 4.064  # Summer on-peak rate in ¢/kWh
    rate_series[~on_peak_hours & (rate_series.index.month == month)] = 2.852  # Summer off-peak rate in ¢/kWh

# Assign rates for non-summer months
for month in non_summer_months:
    on_peak_hours = (rate_series.index.month == month) & (rate_series.index.hour >= 10) & (rate_series.index.hour < 22)
    rate_series[on_peak_hours] = 3.411  # Non-summer on-peak rate in ¢/kWh
    rate_series[~on_peak_hours & (rate_series.index.month == month)] = 2.931  # Non-summer off-peak rate in ¢/kWh

# Convert the rate series to a DataFrame for better visualization
rate_df = rate_series.to_frame(name='Rate (cent/kWh)')
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 30 * 0.05951  # Add a fixed monthly charge
rate_df.drop(columns=['Timestamp'], inplace=True)

# Save the DataFrame to a CSV file
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num] + '.csv'), index=False)

In [ ]:
# 70 Vermont Green Mountain Power Company - Vermont Residential Net Metering (RMP) __ 662c045e28efe948d5045abd
df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\\rates\\CA_southern california edison co_TOU-D-PRIM.csv')

df['Rate (cent/kWh)'] = np.ones(8760) * 19.988
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Remove illegal characters from the filename
cleaned_filename = "".join(c for c in (combined_info[util_num] + '.csv') if c.isalnum() or c in (' ', '.', '_')).rstrip()
cleaned_file_path = os.path.join(rate_folder, cleaned_filename)
df.to_csv(os.path.join(cleaned_file_path), index=False)

In [ ]:
# 67 UT Rocky Mountain Power - Utah_Residential Net Metering Service (A35)__d1b0f2a3c4e5b7c8a9e6f3d4a2e1b5a0
# Define the time range for the year
time_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define the summer and winter months
summer_months = [6, 7, 8, 9]
winter_months = [10, 11, 12, 1, 2, 3, 4, 5]

# Assign rates based on the months
rate_series[rate_series.index.month.isin(summer_months)] = 5.704  # Summer rate in ¢/kWh
rate_series[rate_series.index.month.isin(winter_months)] = 4.199  # Winter rate in ¢/kWh

# Convert the rate series to a DataFrame for better visualization
rate_df = rate_series.to_frame(name='Rate (cent/kWh)')
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 0  # Add a fixed rate column with default value 0

# Save the DataFrame to a CSV file
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num] + '.csv'), index=False)

In [ ]:
# 66 TX Entergy Texas
# Define the time range for the year
time_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define the summer and winter rates
summer_rate = 0.033405 * 100  # Convert to cents per kWh
winter_rate = 0.03499 * 100  # Convert to cents per kWh

# Define the summer and winter months
summer_months = [5, 6, 7, 8, 9]
winter_months = [10, 11, 12, 1, 2, 3, 4]

# Assign rates based on the months
rate_series[rate_series.index.month.isin(summer_months)] = summer_rate
rate_series[rate_series.index.month.isin(winter_months)] = winter_rate

# Convert the rate series to a DataFrame for better visualization
rate_df = rate_series.to_frame(name='Rate (cent/kWh)')
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 0  # Add a fixed rate column with default value 0

# Save the DataFrame to a CSV file
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num] + '.csv'), index=False)

In [ ]:
# 64, 65 TN https://www.tva.com/energy/valley-renewable-energy/dispersed-power-production-program
df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')
monthly_rates = np.array([3.913, 2.704, 1.785, 1.92, 1.783, 2.081, 2.549, 2.536, 1.826, 1.847, 4.376, 3.822])*1.05**2
df['Rate (cent/kWh)'] = np.tile(monthly_rates, 8760 // 12)
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
# Remove illegal characters from the filename
cleaned_filename = "".join(c for c in (combined_info[util_num] + '.csv') if c.isalnum() or c in (' ', '.', '_')).rstrip()
cleaned_file_path = os.path.join(rate_folder, cleaned_filename)
df.to_csv(os.path.join(cleaned_file_path), index=False)

In [ ]:
# 60 RI
# Define the time range for the year
time_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define the rates for each period
jan_mar_rate = 0.16908  # $/kWh
apr_sept_rate = 0.09177  # $/kWh
oct_dec_rate = 0.15187  # $/kWh

# Assign rates based on the months
rate_series[rate_series.index.month.isin([1, 2, 3])] = jan_mar_rate*100
rate_series[rate_series.index.month.isin([4, 5, 6, 7, 8, 9])] = apr_sept_rate*100
rate_series[rate_series.index.month.isin([10, 11, 12])] = oct_dec_rate*100

# Convert the rate series to a DataFrame for better visualization
rate_df = rate_series.to_frame(name='Rate (cent/kWh)')
rate_df['fixed_monthly_charge'] = 5
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)

# Save the DataFrame to a CSV file
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num] + '.csv'), index=False)

In [ ]:
# 50 OH AEP ohio power co
# Define the time range for the year
time_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define the seasonal periods
winter_months = [10, 11, 12, 1, 2, 3, 4, 5]
summer_months = [6, 7, 8, 9]

# Assign rates for winter and summer periods
rate_series[rate_series.index.month.isin(winter_months)] = 2.939  # Winter rate
rate_series[rate_series.index.month.isin(summer_months)] = 2.997  # Summer rate

# Convert the rate series to a DataFrame for better visualization
rate_df = rate_series.to_frame(name='Rate (cent/kWh)')
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 0  # Add a fixed rate column with default value 0
rate_df.drop(columns=['Timestamp'], inplace=True)
rate_df
# Save the DataFrame to a CSV file
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num] + '.csv'), index=False)

In [ ]:
# NV Energy 4.1 flat

df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')
df['Rate (cent/kWh)'] = 4.1
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df
df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
# 44 NM

util_rate = pd.read_csv(basedir + fr'\\PVEV\julia\Utility_structure\{urdb_labels[util_num]}.csv')
util_rate['energy_rates'].unique()
util_rate['peak'] = util_rate['energy_rates'].apply(lambda x: 'peak' if x in [0.179868, 0.320348] else 'off-peak')
util_rate['month'] = pd.to_datetime(util_rate.index, unit='h', origin=pd.Timestamp('2025-01-01')).month
# Map the month names in column 0 to their corresponding numeric values
month_mapping = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
strindf['month'] = strindf[0].map(month_mapping)
# Assign hourly rates based on off-peak and on-peak information
util_rate['Rate (cent/kWh)'] = util_rate.apply(
    lambda row: strindf.loc[strindf['month'] == row['month'], 3].values[0]
    if row['peak'] == 'peak' else strindf.loc[strindf['month'] == row['month'], 4].values[0],
    axis=1
)
util_rate
# util_rate['energy_rates'] = util_rate['energy_rates'].replace(util_rate['energy_rates'].unique(), [2.117, 2.941+0.602, 3.194+3.9])

df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')
df['Rate (cent/kWh)'] = util_rate['Rate (cent/kWh)']*100
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df
df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
from io import StringIO

strin = """
March $0.0032297 $0.0261199 $0.0371072 $0.0145271 $0.0183563 X
April $0.0003868 $0.0229574 $0.0351288 $0.0100913 $0.0146815 X
May $0.0074837 $0.0179934 $0.0415083 $0.0083880 $0.0142642 X
June $0.0313384 $0.0239236 $0.0650987 $0.0228767 $0.0263952 X
July $0.0534161 $0.0361821 $0.0982314 $0.0368525 $0.0422974 X
August $0.0336348 $0.0296482 $0.0460217 $0.0296066 $0.0310628 X
September $0.0321058 $0.0295452 $0.0494262 $0.0265933 $0.0303988 X
October $0.0242631 $0.0314976 $0.0450309 $0.0251208 $0.0288138 X
November $0.0539460 $0.0523322 $0.0668907 $0.0500698 $0.0528694 X
December $0.0326794 $0.0317733 $0.0430293 $0.0298478 $0.0320802 X
January $0.0673754 $0.0801729 $0.0971838 $0.0709834 $0.0756319 X
February $0.0351430 $0.0363440 $0.0637672 $0.0297511 $0.0359092 X"""
# Convert the string into a DataFrame
strindf = pd.read_csv(StringIO(strin), sep=' \$', engine='python', header=None)
strindf

In [ ]:
# 41 NH
# 0.188 flat
# 39 Dominion Energy North Carolina, virginia electric & power co
# all off peak credit 0.09554
df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')

df['Rate (cent/kWh)'] = np.ones(8760)* 18.8
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
# Remove illegal characters from the filename
cleaned_filename = "".join(c for c in (combined_info[util_num] + '.csv') if c.isalnum() or c in (' ', '.', '_')).rstrip()
cleaned_file_path = os.path.join(rate_folder, cleaned_filename)
df.to_csv(os.path.join(cleaned_file_path), index=False)

In [ ]:
# 40 NE
# Define the time range for the year
time_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define the summer months (June to September) and non-summer months (October to May)
summer_months = [6, 7, 8, 9]
non_summer_months = [1, 2, 3, 4, 5, 10, 11, 12]

# Assign rates for summer and non-summer months
rate_series[rate_series.index.month.isin(summer_months)] = 4.00  # Summer rate
rate_series[rate_series.index.month.isin(non_summer_months)] = 3.52  # Non-summer rate

# Convert the rate series to a DataFrame for better visualization
rate_df = rate_series.to_frame(name='Rate (cent/kWh)')
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 0  # Add a fixed rate column with default value 0
rate_df.drop(columns=['Timestamp'], inplace=True)
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
# 38 Dominion Energy North Carolina, virginia electric & power co
# all off peak credit 0.09554
df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')

df['Rate (cent/kWh)'] = np.ones(8760)* 0.09554*100
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
# Remove illegal characters from the filename
cleaned_filename = "".join(c for c in (combined_info[util_num] + '.csv') if c.isalnum() or c in (' ', '.', '_')).rstrip()
cleaned_file_path = os.path.join(rate_folder, cleaned_filename)
df.to_csv(os.path.join(cleaned_file_path), index=False)

In [ ]:
# 37 NC
# Define the time range for the year
time_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define the summer, winter, and shoulder months
summer_months = [6, 7, 8, 9]
winter_months = [12, 1, 2]
shoulder_months = [3, 4, 5, 10, 11]

# Define holidays
holidays = [
    '2025-01-01',  # New Year's Day
    '2025-04-18',  # Good Friday
    '2025-05-26',  # Memorial Day
    '2025-07-04',  # Independence Day
    '2025-09-01',  # Labor Day
    '2025-11-27',  # Thanksgiving Day
    '2025-11-28',  # Day after Thanksgiving
    '2025-12-25',  # Christmas Day
]
holidays = pd.to_datetime(holidays)

# Assign rates based on the defined periods
for timestamp in rate_series.index:
    month = timestamp.month
    weekday = timestamp.weekday()
    hour = timestamp.hour
    is_holiday = timestamp.normalize() in holidays or \
                 (timestamp.weekday() == 4 and timestamp.month == 12 and timestamp.day == 26) or \
                 (timestamp.weekday() == 0 and timestamp.month == 12 and timestamp.day == 29)

    if is_holiday or weekday >= 5:  # Off-peak for holidays and weekends
        if month in summer_months:
            rate_series[timestamp] = 3.41
        elif month in winter_months:
            rate_series[timestamp] = 3.79
        elif month in shoulder_months:
            rate_series[timestamp] = 3.03
    else:
        if month in summer_months:
            if 14 <= hour < 18 or 22 <= hour < 24:  # Summer on-peak
                rate_series[timestamp] = 4.14
            elif 18 <= hour < 22:  # Summer premium peak
                rate_series[timestamp] = 4.67
            else:  # Summer off-peak
                rate_series[timestamp] = 3.41
        elif month in winter_months:
            if 4 <= hour < 6:  # Winter morning on-peak
                rate_series[timestamp] = 4.21
            elif 17 <= hour < 23:  # Winter evening on-peak
                rate_series[timestamp] = 4.60
            elif 6 <= hour < 9:  # Winter premium peak
                rate_series[timestamp] = 5.86
            else:  # Winter off-peak
                rate_series[timestamp] = 3.79
        elif month in shoulder_months:
            if 5 <= hour < 9 or 17 <= hour < 23:  # Shoulder on-peak
                rate_series[timestamp] = 3.79
            else:  # Shoulder off-peak
                rate_series[timestamp] = 3.03

# Convert the rate series to a DataFrame for better visualization
rate_df = rate_series.to_frame(name='Rate (cent/kWh)')
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 0  # Add a fixed rate column with default value 0
rate_df.drop(columns=['Timestamp'], inplace=True)
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
# 34 MS_mississippi power co
df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')

df['Rate (cent/kWh)'] = np.ones(8760)* 5.5
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
# Remove illegal characters from the filename
cleaned_filename = "".join(c for c in (combined_info[util_num] + '.csv') if c.isalnum() or c in (' ', '.', '_')).rstrip()
cleaned_file_path = os.path.join(rate_folder, cleaned_filename)
df.to_csv(os.path.join(cleaned_file_path), index=False)

In [ ]:
# 32 MO_union electric co avoided cost rate 
df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')

df['Rate (cent/kWh)'] = np.ones(8760)* 7.3
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
# Remove illegal characters from the filename
cleaned_filename = "".join(c for c in (combined_info[util_num] + '.csv') if c.isalnum() or c in (' ', '.', '_')).rstrip()
cleaned_file_path = os.path.join(rate_folder, cleaned_filename)
df.to_csv(os.path.join(cleaned_file_path), index=False)

In [ ]:
# 30 minnesota, 40 ND, 64
# net billing

util_rate = pd.read_csv(basedir + fr'\\PVEV\julia\Utility_structure\{urdb_labels[util_num]}.csv')
util_rate
util_rate['energy_rates'] = util_rate['energy_rates'].replace(util_rate['energy_rates'].unique(), [2.117, 2.941+0.602, 3.194+3.9])

df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')
df['Rate (cent/kWh)'] = util_rate['energy_rates']
df['fixed_monthly_charge'] = 5
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df
df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
util_rate

In [ ]:
util_rate['energy_rates'].unique()

In [ ]:
# 30 MO 0.033 flat rate
df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')

df['Rate (cent/kWh)'] = np.ones(8760)* 3.3
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df
df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
# 29 michigan
# net billing
util_rate = pd.read_csv(basedir + fr'\\PVEV\julia\Utility_structure\{urdb_labels[util_num]}.csv')
# Group the data into chunks of 24 rows and find the minimum value in each chunk
# min_values = util_rate['energy_rates'].groupby(util_rate.index // 24).transform('min')

# Replace each 24-row chunk with its minimum value
# util_rate['energy_rates'] = min_values
util_rate['energy_rates'] = util_rate['energy_rates'].replace(util_rate['energy_rates'].unique(), [5.14, 13.422, 5.35, 15.89])

df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')
df['Rate (cent/kWh)'] = util_rate['energy_rates']
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
# 24 LA_entergy louisiana inc
# net billing avoided cost
# Define the time range for the year
time_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define on-peak and off-peak rates
on_peak_rate = 2.59331  # ¢ per kWh
off_peak_rate = 2.59331  # ¢ per kWh

# Assign rates based on the defined periods
for timestamp in rate_series.index:
    if (timestamp.weekday() < 5) and (7 <= timestamp.hour < 21):  # Monday to Friday, 7 a.m. to 9 p.m.
        rate_series[timestamp] = on_peak_rate
    else:  # Off-peak hours
        rate_series[timestamp] = off_peak_rate

# Convert the rate series to a DataFrame for better visualization
rate_df = rate_series.to_frame(name='Rate (cent/kWh)')
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 0  # Add a fixed rate column with default value 0
rate_df.drop(columns=['Timestamp'], inplace=True)
rate_df
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
# 20 indiana michigan power
# net billing
# Define the time range for the year
time_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define on-peak and off-peak rates
on_peak_rate = 4.41  # ¢ per kWh
off_peak_rate = 3.07  # ¢ per kWh

# Assign rates based on the defined periods
for timestamp in rate_series.index:
    if (timestamp.weekday() < 5) and (7 <= timestamp.hour < 21):  # Monday to Friday, 7 a.m. to 9 p.m.
        rate_series[timestamp] = on_peak_rate
    else:  # Off-peak hours
        rate_series[timestamp] = off_peak_rate

# Convert the rate series to a DataFrame for better visualization
rate_df = rate_series.to_frame(name='Rate (cent/kWh)')
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 0.5  # Add a fixed rate column with default value 0
rate_df.drop(columns=['Timestamp'], inplace=True)
rate_df
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
# 15 ID
# net billing
util_rate = pd.read_csv(basedir + fr'\\PVEV\julia\Utility_structure\{urdb_labels[util_num]}.csv')
# Define the time range for the year
time_index = pd.date_range(start='2023-01-01 00:00:00', end='2023-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define the summer months (June to September) and non-summer months (October to May)
summer_months = [6, 7, 8, 9]
non_summer_months = [1, 2, 3, 4, 5, 10, 11, 12]

# Define holidays (example: New Year's Day, Independence Day, etc.)
holidays = [
    '2023-01-01',  # New Year's Day
    '2023-07-04',  # Independence Day
    '2023-12-25',  # Christmas Day
]
holidays = pd.to_datetime(holidays)

# Assign rates for summer months
for month in summer_months:
    on_peak_hours = (
        (rate_series.index.month == month) &
        (rate_series.index.weekday < 6) &  # Monday to Saturday
        (~rate_series.index.normalize().isin(holidays)) &  # Exclude holidays
        (rate_series.index.hour >= 15) & (rate_series.index.hour < 23)  # 3:00 p.m. to 11:00 p.m.
    )
    rate_series[on_peak_hours] = 16.9966  # Summer On-Peak
    rate_series[~on_peak_hours & (rate_series.index.month == month)] = 5.6533  # Summer Off-Peak

# Assign rates for non-summer months
for month in non_summer_months:
    rate_series[rate_series.index.month == month] = 4.8365  # Non-Summer Flat Rate
rate_series = rate_series.to_frame(name='Rate (cent/kWh)')
rate_series.reset_index(inplace=True)
rate_series.rename(columns={'index': 'Timestamp'}, inplace=True)
df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')
df['Rate (cent/kWh)'] = rate_series['Rate (cent/kWh)']
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df
df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)


In [ ]:
rate_series

In [ ]:
# 12 iowa
# net billing
util_rate = pd.read_csv(basedir + fr'\\PVEV\julia\Utility_structure\{urdb_labels[util_num]}.csv')
# Group the data into chunks of 24 rows and find the minimum value in each chunk
# min_values = util_rate['energy_rates'].groupby(util_rate.index // 24).transform('min')

# Replace each 24-row chunk with its minimum value
# util_rate['energy_rates'] = min_values
util_rate['energy_rates'] = util_rate['energy_rates'].replace(util_rate['energy_rates'].unique(), [5.08, 0, 2.44,4.18])

df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')
df['Rate (cent/kWh)'] = util_rate['energy_rates']
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
df

In [ ]:
# 11 georgia
# all excess paid at low tou rate
util_rate = pd.read_csv(basedir + fr'\\PVEV\julia\Utility_structure\{urdb_labels[util_num]}.csv')
# Group the data into chunks of 24 rows and find the minimum value in each chunk
min_values = util_rate['energy_rates'].groupby(util_rate.index // 24).transform('min')

# Replace each 24-row chunk with its minimum value
util_rate['energy_rates'] = min_values

df = pd.read_csv(basedir + r'\\PVEV\\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')
df['Rate (cent/kWh)'] = util_rate['energy_rates']*100
df['fixed_monthly_charge'] = 0
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)


In [ ]:
# Define the time range for the year
time_index = pd.date_range(start='2023-01-01 00:00:00', end='2023-12-31 23:00:00', freq='H')

# Create a constant rate series
rate_df = pd.Series(0.06857, index=time_index, name='Rate (cent/kWh)')

# Convert the series to a DataFrame for better visualization
rate_df = rate_df.to_frame()
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
# rate_df['fixed_monthly_charge'] = 0.9  # Add a fixed rate column with default value 0.0
rate_df.drop(columns=['Timestamp'], inplace=True)
rate_df['fixed_monthly_charge'] = 0  # Add a fixed rate column with default value 0.0
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)
# Save the DataFrame to a CSV file
# constant_rate_df.to_csv(os.path.join(rate_folder, 'constant_rate_2023.csv'), index=False)

In [ ]:
import pandas as pd
import numpy as np

# 0
# Define the time range for the year
time_index = pd.date_range(start='2023-01-01 00:00:00', end='2023-12-31 23:00:00', freq='H')

# Initialize the rate series
rate_series = pd.Series(index=time_index, dtype=float)

# Define the rates
summer_peak_rate = 4.69  # ¢ per kWh
summer_offpeak_rate = 3.35  # ¢ per kWh
winter_peak_rate = 3.53  # ¢ per kWh
winter_offpeak_rate = 3.35  # ¢ per kWh

# Define the summer months (June to September) and winter months (October to May)
summer_months = [6, 7, 8, 9]
winter_months = [10, 11, 12, 1, 2, 3, 4, 5]

# Assign rates for summer months
for month in summer_months:
    peak_hours = (rate_series.index.month == month) & (rate_series.index.weekday < 5) & (rate_series.index.hour >= 10) & (rate_series.index.hour < 21)
    rate_series[peak_hours] = summer_peak_rate
    rate_series[~peak_hours & (rate_series.index.month == month)] = summer_offpeak_rate

# Assign rates for winter months
for month in winter_months:
    peak_hours = (rate_series.index.month == month) & (rate_series.index.weekday < 5) & (rate_series.index.hour >= 7) & (rate_series.index.hour < 21)
    rate_series[peak_hours] = winter_peak_rate
    rate_series[~peak_hours & (rate_series.index.month == month)] = winter_offpeak_rate

# Convert the rate series to a DataFrame for better visualization
rate_df = rate_series.to_frame(name='Rate (cent/kWh)')
rate_df.reset_index(inplace=True)
rate_df.rename(columns={'index': 'Timestamp'}, inplace=True)
rate_df['fixed_monthly_charge'] = 0.9  # Add a fixed rate column with default value 0.0
rate_df.drop(columns=['Timestamp'], inplace=True)
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
# DC 8
util_rate = pd.read_csv(r'E:\University of Michigan Dropbox\Jiahui Chen\PVEV\julia\Utility_structure\67a62b6276ceb20ae0055a6b.csv')
rate = util_rate['energy_rates'].replace({0.13999: 8.581, 0.11195: 5.777,0.27867:22.449,0.29624:24.206}, regex=True).astype(float) 
rate_df['Rate (cent/kWh)'] = rate
rate_df['fixed_monthly_charge'] = 0
rate_df.to_csv(os.path.join(rate_folder, combined_info[util_num]+'.csv'), index=False)

In [ ]:
util_rate['energy_rates'][0:24]